# 4. Deployment Strategy - Interactive Phishing Detection Demo

Building a production-ready demonstration of our hybrid phishing detection system using Streamlit.

## 1. Deployment Overview

**What we're building:**

An interactive phishing detection demo using Streamlit that demonstrates our hybrid approach on dataset4 URLs.

**Demo Mode approach:**
- Select URLs from dataset4 (pre-extracted features)
- Run through hybrid system: Rules → XGBoost
- Show prediction vs actual label
- Display feature breakdown and explanation

**Why Demo Mode (not Live URL scanning)?**

**Live Mode would require:**
1. Fetching live webpages (`requests.get(url)`)
2. Extracting ALL 49 features from raw HTML:
   - Easy features: URLLength, IsHTTPS, NoOfJS (count `<script>` tags)
   - Hard features: TLDLegitimateProb (need TLD reputation database), CharContinuationRate (complex calculation), DomainTitleMatchScore (fuzzy matching)
3. Handling failures: Timeouts, CAPTCHAs, blocked requests, malformed HTML
4. Building 49 feature extractors (significant engineering effort)

**Why we chose Demo Mode:**
- **Focus on ML, not web scraping**: This project demonstrates machine learning for phishing detection, not production web crawling
- **Reliability**: Demo always works (no network issues, timeouts, blocked requests)
- **Assessment-friendly**: Reproducible results for grading
- **Proof of concept**: Shows model performance without production engineering complexity

**In production:** Feature extraction would be handled by a dedicated service (similar to Google Safe Browsing's web crawlers). Our model consumes features, not raw URLs.

**Tech stack:**
- Streamlit (interactive UI)
- XGBoost (pre-trained model from notebook 3)
- pandas (feature handling)

## 2. Model Preparation

**Goal:** Train and save XGBoost model for deployment.

**Why train on full dataset:**

In notebook 3, we used 80/20 split to EVALUATE XGBoost performance (99.995% recall achieved). 

Now for deployment, we retrain on ALL 235,795 URLs to give the model maximum training data. This is standard ML practice:
- **Evaluation phase**: Use split to test performance
- **Deployment phase**: Retrain on all data for strongest model

We are NOT re-testing - we accept the circular validation limitation acknowledged in notebook 3.

**Steps:**
1. Load dataset4 and prepare features (49 numeric features)
2. Train XGBoost on full dataset
3. Save model to `models/xgb_model.pkl`
4. Validate loading works

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from xgboost import XGBClassifier

# Load dataset4
df = pd.read_csv('data/dataset4.csv')

print(f"Total URLs: {len(df):,}")
print(f"Phishing: {(df['label']==0).sum():,} ({(df['label']==0).sum()/len(df)*100:.1f}%)")
print(f"Legitimate: {(df['label']==1).sum():,} ({(df['label']==1).sum()/len(df)*100:.1f}%)")

# Prepare features (same as notebook 3)
features_to_exclude = ['URLSimilarityIndex', 'FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label']
feature_cols = [col for col in df.columns if col not in features_to_exclude]

X_full = df[feature_cols]
y_full = df['label']

print(f"\nUsing {len(feature_cols)} features for training")

In [ ]:
# Train XGBoost on full dataset
xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

print("Training XGBoost on full dataset (235,795 URLs)...")
xgb_model.fit(X_full, y_full)
print("Training complete!")

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save model
model_path = 'models/xgb_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(xgb_model, f)

print(f"\nModel saved to: {model_path}")
print(f"Model file size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

In [ ]:
# Validate: Load model and test prediction
print("Validating model loading...")

with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Test on first 5 URLs
test_sample = X_full.head(5)
predictions = loaded_model.predict(test_sample)
probabilities = loaded_model.predict_proba(test_sample)

print("\nTest predictions on first 5 URLs:")
for i in range(5):
    actual = "Phishing" if y_full.iloc[i] == 0 else "Legitimate"
    pred = "Phishing" if predictions[i] == 0 else "Legitimate"
    prob_phishing = probabilities[i][0] * 100
    prob_legitimate = probabilities[i][1] * 100
    
    print(f"URL {i+1}: Actual={actual}, Predicted={pred}, Prob(Phishing)={prob_phishing:.1f}%, Prob(Legitimate)={prob_legitimate:.1f}%")

print("\n✓ Model loading and prediction working correctly!")